In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import sys
sys.path.append("../../")

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import rateslib as rl

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

In [2]:
cache_path = r"C:\Users\chris\clee\project-oasis\private\sdranalytics\.cache"

# as_of = datetime.date(2026, 2, 23)
# start = NY_tz.localize(datetime.datetime(as_of.year, as_of.month, as_of.day, 0, 0))
# end = NY_tz.localize(datetime.datetime(as_of.year, as_of.month, as_of.day, 23, 59))

start = NY_tz.localize(datetime.datetime(2026, 4, 23, 0, 0))
end = NY_tz.localize(datetime.datetime(2026, 4, 23, 23, 59))

# mdp = IRSwapsMDP(source="ERIS_EOD_LIVE-QL_BASIC")
# pricer = mdp.get_pricer(request=dict(curve_name="USD-SOFR-1D", timestamp=start.date()))

from SDRUtils.data.builder import SDRDataBuilder
sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=True)
df = sdr.grab_sdr_trades(
	start_timestamp=start,
	end_timestamp=end,
	agency="CFTC",
	asset_class="RATES",
)
df

MERGING SLICES...: 100%|██████████| 2/2 [00:00<00:00, 121.18it/s]


,Dissemination Identifier,Original Dissemination Identifier,Action type,Event type,Event timestamp,Amendment indicator,Asset Class,Product name,Cleared,Mandatory clearing indicator,...,Package transaction price notation,Package transaction spread,Package transaction spread currency,Package transaction spread notation,Physical delivery location-Leg 1,Delivery Type,Unique Product Identifier,UPI FISN,UPI Underlier Name,file_date
0,2828935377000000101,2817893246000000901,MODI,TRAD,2026-04-23 04:00:05+00:00,False,IR,None,N,False,...,NaN,,,NaN,None,None,QZSG7CBVPLPR,NA/O Call Epn Fxd Flt KRW,NA/Swap Fxd Flt KRW,2026-04-23
1,2829132280000000601,,NEWT,TRAD,2026-04-23 04:00:11+00:00,False,IR,None,N,False,...,NaN,,,NaN,None,None,QZ4HTNF4RVB5,NA/Swap Flt Flt AUD USD,AUD-BBSW vs USD-SOFR-OIS Compound,2026-04-23
2,2830909814000001101,,NEWT,TRAD,2026-04-23 04:00:11+00:00,False,IR,None,N,False,...,NaN,,,NaN,None,None,QZ4HTNF4RVB5,NA/Swap Flt Flt AUD USD,AUD-BBSW vs USD-SOFR-OIS Compound,2026-04-23
3,2828934923000000301,2817449205000002701,MODI,TRAD,2026-04-23 04:00:12+00:00,False,IR,None,N,False,...,NaN,,,NaN,None,None,QZSG7CBVPLPR,NA/O Call Epn Fxd Flt KRW,NA/Swap Fxd Flt KRW,2026-04-23
4,2828946362000000101,,NEWT,TRAD,2026-04-23 04:00:24+00:00,None,IR,None,I,False,...,NaN,,,NaN,None,None,QZPFCRJXSDC9,NA/Swap OIS INR,INR-MIBOR-OIS Compound,2026-04-23
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23016,2846255235000000101,,NEWT,TRAD,2026-04-24 03:51:25+00:00,None,IR,None,I,False,...,NaN,NaN,None,NaN,None,None,QZPB1RXQ05RR,NA/Swap OIS INR,INR-MIBOR-OIS-COMPOUND,NaN
23017,2846260852000000101,2846206674000000101,CORR,,2026-04-24 03:53:37+00:00,None,IR,None,N,False,...,NaN,NaN,None,NaN,None,None,QZ26BPG38C1K,NA/Swap Fxd Flt KRW,KRW-CD 91D,NaN
23018,2846261272000000101,,NEWT,TRAD,2026-04-24 03:54:01+00:00,None,IR,None,I,False,...,NaN,NaN,None,NaN,None,None,QZPB1RXQ05RR,NA/Swap OIS INR,INR-MIBOR-OIS-COMPOUND,NaN
23019,2846261273000000201,,NEWT,TRAD,2026-04-24 03:54:12+00:00,None,IR,None,I,False,...,1.0,NaN,None,NaN,None,None,QZB883814F13,NA/Swap OIS INR,INR-MIBOR-OIS Compound,NaN


In [7]:
df["Cleared"].value_counts()

Cleared
I    20618
N     5594
        45
Y        5
Name: count, dtype: int64

In [21]:
# df["Event timestamp"] = df["Event timestamp"].astype(str)
# df["Execution Timestamp"] = df["Execution Timestamp"].astype(str)
# df.to_csv(r"C:\Users\chris\clee\ARBS\notebooks\sdr\april_fomc_dated_sdr_trades.csv", index=False)	

In [3]:
# df[(df["Effective Date"].dt.date == datetime.date(2026, 4, 28)) & (df["Expiration Date"].dt.date == datetime.date(2026, 6, 16))]

In [3]:
df[df["Dissemination Identifier"] == "2844284325000000101"].iloc[0].to_dict()

{'Dissemination Identifier': '2844284325000000101',
 'Original Dissemination Identifier': '',
 'Action type': 'NEWT',
 'Event type': 'TRAD',
 'Event timestamp': Timestamp('2026-04-23 20:33:56+0000', tz='UTC'),
 'Amendment indicator': None,
 'Asset Class': 'IR',
 'Product name': None,
 'Cleared': 'I',
 'Mandatory clearing indicator': True,
 'Execution Timestamp': Timestamp('2026-04-23 20:33:56+0000', tz='UTC'),
 'Effective Date': Timestamp('2026-07-06 00:00:00'),
 'Expiration Date': Timestamp('2028-03-15 00:00:00'),
 'Maturity date of the underlier': None,
 'Non-standardized term indicator': None,
 'Platform identifier': 'BILT',
 'Prime brokerage transaction indicator': False,
 'Block trade election indicator': False,
 'Large notional off-facility swap election indicator': True,
 'Notional amount-Leg 1': '1,100,000,000+',
 'Notional amount-Leg 2': '1,100,000,000+',
 'Notional currency-Leg 1': 'USD',
 'Notional currency-Leg 2': 'USD',
 'Notional quantity-Leg 1': None,
 'Notional quantity

In [23]:
df[
    # (df["Package transaction spread"].notna())
    # & (df["UPI Underlier Name"] == "USD-SOFR-OIS Compound")
    (df["Expiration Date"].dt.date == datetime.date(2056, 2, 15))
]

,Dissemination Identifier,Original Dissemination Identifier,Action type,Event type,Event timestamp,Amendment indicator,Asset Class,Product name,Cleared,Mandatory clearing indicator,...,Package transaction price notation,Package transaction spread,Package transaction spread currency,Package transaction spread notation,Physical delivery location-Leg 1,Delivery Type,Unique Product Identifier,UPI FISN,UPI Underlier Name,file_date
5207,2662959014000000401,,NEWT,TRAD,2026-04-09 08:39:12+00:00,None,IR,None,I,True,...,3.0,,,NaN,None,None,QZPB5VSBGRCD,NA/Swap OIS USD,USD-SOFR-OIS Compound,2026-04-09
8515,2669096871000000201,2668734155000000601,MODI,TRAD,2026-04-09 11:25:13+00:00,False,IR,None,I,False,...,1.0,,,NaN,None,None,QZPB5VSBGRCD,NA/Swap OIS USD,USD-SOFR-OIS Compound,2026-04-09
8516,2668734155000000601,,NEWT,TRAD,2026-04-09 11:25:13+00:00,None,IR,None,I,False,...,1.0,,,NaN,None,None,QZPB5VSBGRCD,NA/Swap OIS USD,USD-SOFR-OIS Compound,2026-04-09
8572,2667333845000000101,,NEWT,TRAD,2026-04-09 11:26:48+00:00,None,IR,None,I,True,...,NaN,,,NaN,None,None,QZPB5VSBGRCD,NA/Swap OIS USD,USD-SOFR-OIS Compound,2026-04-09
9842,2668275579000000101,,NEWT,TRAD,2026-04-09 12:23:27+00:00,None,IR,None,I,True,...,NaN,,,NaN,None,None,QZPB5VSBGRCD,NA/Swap OIS USD,USD-SOFR-OIS Compound,2026-04-09
13825,2670184353000001201,,NEWT,TRAD,2026-04-09 14:22:25+00:00,None,IR,None,I,True,...,NaN,,,NaN,None,None,QZPB5VSBGRCD,NA/Swap OIS USD,USD-SOFR-OIS Compound,2026-04-09
13991,2670372608000000101,,NEWT,TRAD,2026-04-09 14:25:09+00:00,None,IR,None,I,True,...,NaN,,,NaN,None,None,QZPB5VSBGRCD,NA/Swap OIS USD,USD-SOFR-OIS Compound,2026-04-09
14388,2670327344000000701,,NEWT,TRAD,2026-04-09 14:33:24+00:00,None,IR,None,I,True,...,NaN,,,NaN,None,None,QZPB5VSBGRCD,NA/Swap OIS USD,USD-SOFR-OIS Compound,2026-04-09
14790,2670427385000000401,2670184353000001201,TERM,ETRM,2026-04-09 14:41:58+00:00,None,IR,None,I,None,...,NaN,,,NaN,None,None,QZPB5VSBGRCD,NA/Swap OIS USD,USD-SOFR-OIS Compound,2026-04-09
14791,2670427384000000301,,NEWT,TRAD,2026-04-09 14:41:58+00:00,None,IR,None,I,True,...,NaN,,,NaN,None,None,QZPB5VSBGRCD,NA/Swap OIS USD,USD-SOFR-OIS Compound,2026-04-09
